In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

import pandas as pd

# load just the datasets q01 needs.
factor = 1
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON


def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

# Filter lineitem and extract unique order keys
fline_keys = lineitem.loc[lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE, 'L_ORDERKEY']

# Filter orders by date range
orders_filtered = orders.loc[
    (orders.O_ORDERDATE >= '1993-08-01') &
    (orders.O_ORDERDATE < '1993-11-01')
]

# Select orders matching the filtered lineitem keys
matched_orders = orders_filtered.loc[
    orders_filtered.O_ORDERKEY.isin(fline_keys)
]

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

total = matched_orders.groupby('O_ORDERPRIORITY', as_index=False)['O_ORDERKEY'].count()